# 03. Очищення і метрики
**Мета:** порахувати тривалість етапів і перевірити якість даних.

**Вхід:** `data/processed/candidates.parquet`.

**Що робимо:** дні від оголошення до переможця, від переможця до договору і загалом; дублікати, пропуски, від'ємні інтервали; тендери понад 365 днів позначаємо (`is_long`), а не видаляємо; перші медіани і перцентилі.

**Вихід:** `data/processed/clean.parquet` (9 305 тендерів з повним циклом).

In [1]:
import duckdb
con = duckdb.connect()
cand = '../data/processed/candidates.parquet'

con.sql(f"""
    CREATE OR REPLACE TABLE t AS
    SELECT *,
        date_diff('day', tender_start_date, award_date)     AS days_to_award,
        date_diff('day', award_date,        published_date) AS days_award_to_contract,
        date_diff('day', tender_start_date, published_date) AS days_total
    FROM '{cand}'
    WHERE procurement_method IN ('open', 'selective')
""")

con.sql("""
    SELECT category, procurement_method,
           COUNT(*) AS n,
           SUM(CASE WHEN days_total < 0 THEN 1 ELSE 0 END) AS n_negative,
           MIN(days_total) AS min_d,
           quantile_cont(days_total, 0.5)  AS median_d,
           quantile_cont(days_total, 0.9)  AS p90_d,
           MAX(days_total) AS max_d
    FROM t
    GROUP BY ALL
    ORDER BY category, procurement_method
""").show()

┌──────────────────────┬────────────────────┬───────┬────────────┬───────┬──────────┬────────────────────┬───────┐
│       category       │ procurement_method │   n   │ n_negative │ min_d │ median_d │       p90_d        │ max_d │
│       varchar        │      varchar       │ int64 │   int128   │ int64 │  double  │       double       │ int64 │
├──────────────────────┼────────────────────┼───────┼────────────┼───────┼──────────┼────────────────────┼───────┤
│ БпЛА                 │ open               │  2925 │          0 │     4 │     20.0 │               28.0 │   718 │
│ БпЛА                 │ selective          │  5195 │          0 │     3 │      9.0 │               16.0 │    82 │
│ Керування і зв'язок  │ open               │    49 │          0 │     6 │     19.0 │               26.0 │    34 │
│ РЕБ                  │ open               │  1054 │          0 │     6 │     21.0 │               29.0 │    61 │
│ Розвідка і виявлення │ open               │    82 │          0 │     8 │     1

In [2]:
con.sql("""
    SELECT
        COUNT(*)                                                    AS n,
        COUNT(DISTINCT tender_id)                                   AS n_unique,
        COUNT(*) - COUNT(days_total)                                AS n_null_total,
        SUM(CASE WHEN days_to_award < 0 THEN 1 ELSE 0 END)          AS neg_to_award,
        SUM(CASE WHEN days_award_to_contract < 0 THEN 1 ELSE 0 END) AS neg_award_to_contract,
        SUM(CASE WHEN days_total > 365 THEN 1 ELSE 0 END)           AS n_over_year,
        SUM(CASE WHEN tender_start_date < '2022-01-01' THEN 1 ELSE 0 END) AS n_before_2022
    FROM t
""").show()

┌───────┬──────────┬──────────────┬──────────────┬───────────────────────┬─────────────┬───────────────┐
│   n   │ n_unique │ n_null_total │ neg_to_award │ neg_award_to_contract │ n_over_year │ n_before_2022 │
│ int64 │  int64   │    int64     │    int128    │        int128         │   int128    │    int128     │
├───────┼──────────┼──────────────┼──────────────┼───────────────────────┼─────────────┼───────────────┤
│  9305 │     9305 │            0 │            0 │                     0 │           1 │             0 │
└───────┴──────────┴──────────────┴──────────────┴───────────────────────┴─────────────┴───────────────┘



In [4]:
con.sql("""
    SELECT tender_id, category, procurement_method, days_to_award, days_award_to_contract, days_total
    FROM t
    WHERE days_total > 365
""").show()

┌──────────────────────────────────┬──────────┬────────────────────┬───────────────┬────────────────────────┬────────────┐
│            tender_id             │ category │ procurement_method │ days_to_award │ days_award_to_contract │ days_total │
│             varchar              │ varchar  │      varchar       │     int64     │         int64          │   int64    │
├──────────────────────────────────┼──────────┼────────────────────┼───────────────┼────────────────────────┼────────────┤
│ e8814dbbcbf84c5e846371f46ca02622 │ БпЛА     │ open               │            38 │                    680 │        718 │
└──────────────────────────────────┴──────────┴────────────────────┴───────────────┴────────────────────────┴────────────┘



In [5]:
con.sql("""
    SELECT tender_id,
           number_of_awards,
           has_cancelled_awards,
           has_unsuccessful_awards,
           number_of_contracts,
           tender_start_date,
           award_date,
           published_date
    FROM t
    WHERE tender_id = 'e8814dbbcbf84c5e846371f46ca02622'
""").show()

┌──────────────────────────────────┬──────────────────┬──────────────────────┬─────────────────────────┬─────────────────────┬───────────────────────────────┬───────────────────────────────┬───────────────────────────────┐
│            tender_id             │ number_of_awards │ has_cancelled_awards │ has_unsuccessful_awards │ number_of_contracts │       tender_start_date       │          award_date           │        published_date         │
│             varchar              │      int64       │        int64         │          int64          │        int64        │   timestamp with time zone    │   timestamp with time zone    │   timestamp with time zone    │
├──────────────────────────────────┼──────────────────┼──────────────────────┼─────────────────────────┼─────────────────────┼───────────────────────────────┼───────────────────────────────┼───────────────────────────────┤
│ e8814dbbcbf84c5e846371f46ca02622 │                1 │                    0 │                       0 │    

In [6]:
con.sql("""
    COPY (SELECT *, days_total > 365 AS is_long FROM t)
    TO '../data/processed/clean.parquet' (FORMAT parquet)
""")

con.sql("SELECT COUNT(*) FROM '../data/processed/clean.parquet'").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         9305 │
└──────────────┘



In [7]:
import duckdb
con = duckdb.connect()
clean = '../data/processed/clean.parquet'

q1 = con.sql(f"""
    SELECT category, procurement_method,
           COUNT(*) AS n,
           quantile_cont(days_to_award, 0.5)          AS award_p50,
           quantile_cont(days_award_to_contract, 0.5) AS contract_p50,
           quantile_cont(days_total, 0.5)             AS total_p50,
           quantile_cont(days_total, 0.8)             AS total_p80,
           quantile_cont(days_total, 0.9)             AS total_p90
    FROM '{clean}'
    GROUP BY ALL
    ORDER BY category, procurement_method
""").df()
q1

,category,procurement_method,n,award_p50,contract_p50,total_p50,total_p80,total_p90
0,БпЛА,open,2925,11.0,8.0,20.0,25.0,28.0
1,БпЛА,selective,5195,5.0,4.0,9.0,13.0,16.0
2,Керування і зв'язок,open,49,9.0,7.0,19.0,24.0,26.0
3,РЕБ,open,1054,12.0,8.0,21.0,27.0,29.0
4,Розвідка і виявлення,open,82,9.5,7.0,18.0,21.8,23.9
